# Notebook 3 - Heartbeats, Truncation, and Real-World HWMs

The first two notebooks introduced the high-water mark. This one fills in the pieces a real system needs:

1. **How followers learn the HWM** - via heartbeats from the leader.
2. **Truncation** - what a follower does when it reconnects with extra entries beyond the new leader's HWM.
3. **Reading from followers safely** - the same rule (`offset <= HWM`) protects follower reads too.
4. **Where you meet HWMs in production** - Kafka, Raft, MongoDB, PostgreSQL, ZooKeeper.

We will build a slightly richer simulation that exchanges *messages* between nodes instead of mutating shared state directly.


## Setup

```bash
cd 02-distributed-primitives/high-water-mark
uv sync
```

Pick the `.venv` kernel in VS Code, reload the window if needed.


## A tiny replicated log with heartbeats

We model three message types - just enough to see HWM propagation:

- `AppendEntries(offset, entry, leader_hwm)` - leader to follower, ships an entry **and** the leader's current HWM.
- `Ack(follower, offset)` - follower to leader, "I durably stored up to this offset".
- `Heartbeat(leader_hwm)` - leader to follower, periodic update of the HWM (used when there are no new entries).

Each follower keeps its own `commit_index` (its local view of the HWM) and only exposes `log[: commit_index + 1]` to clients.


In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional

@dataclass
class Follower:
    name: str
    log: List[str] = field(default_factory=list)
    commit_index: int = -1  # follower's known HWM

    def receive_append(self, offset: int, entry: str, leader_hwm: int) -> Optional[int]:
        # Only accept the entry if it lines up with our log.
        if offset == len(self.log):
            self.log.append(entry)
        elif offset < len(self.log):
            # We already hold something at this offset. Truncate ONLY on a genuine
            # conflict. Blindly deleting log[offset+1:] on every AppendEntries would
            # discard entries we already have and the leader also has, which is both
            # wasteful and — if the leader crashes mid-catch-up — data loss.
            # (Real Raft compares (index, term); we have no terms, so we compare the
            # entry itself, which is the same idea at this level of detail.)
            if self.log[offset] != entry:
                self.log[offset] = entry
                del self.log[offset + 1:]      # divergent suffix, drop it
        else:
            # Gap - refuse, leader will re-send earlier entries.
            return None
        # Followers can advance their commit_index up to what the leader says,
        # but never past their own log length.
        self.commit_index = min(leader_hwm, len(self.log) - 1)
        return offset  # ack offset

    def heartbeat(self, leader_hwm: int) -> None:
        self.commit_index = min(leader_hwm, len(self.log) - 1)

    def truncate_to(self, log_end: int) -> int:
        """Drop everything at or beyond `log_end`. Returns how many entries went.

        A rejoining node does this BEFORE it starts fetching from the new leader.
        It is a distinct step, not a side effect of replication: re-sending entries
        the follower already has must never delete anything (see receive_append),
        so somebody has to say explicitly "your log is too long".
        """
        dropped = max(0, len(self.log) - log_end)
        del self.log[log_end:]
        self.commit_index = min(self.commit_index, len(self.log) - 1)
        return dropped

    def visible_to_clients(self) -> List[str]:
        return self.log[: self.commit_index + 1]


@dataclass
class Leader:
    name: str = "L"
    log: List[str] = field(default_factory=list)
    match_index: Dict[str, int] = field(default_factory=dict)
    quorum: int = 2

    def append(self, entry: str) -> int:
        self.log.append(entry)
        return len(self.log) - 1

    @property
    def high_water_mark(self) -> int:
        leader_offset = len(self.log) - 1
        offsets = sorted([leader_offset, *self.match_index.values()], reverse=True)
        # If we do not yet have enough nodes (leader + acks) to form a quorum,
        # nothing is committed yet.
        if len(offsets) < self.quorum:
            return -1
        return offsets[self.quorum - 1]

    def replicate_to(self, follower: Follower, offset: int) -> None:
        ack = follower.receive_append(offset, self.log[offset], self.high_water_mark)
        if ack is not None:
            self.match_index[follower.name] = max(self.match_index.get(follower.name, -1), ack)

    def heartbeat_to(self, follower: Follower) -> None:
        follower.heartbeat(self.high_water_mark)


## Step through a normal cycle

Watch how the HWM propagates: it is *one round-trip behind* the data. After the leader replicates entry `N`, a follower only learns that `N` is committed on the **next** message - either the next `AppendEntries` or a `Heartbeat`.


In [ ]:
leader = Leader()
f1, f2 = Follower("f1"), Follower("f2")

def step(label, fn):
    fn()
    print(
        f"{label:<28}  hwm={leader.high_water_mark}  "
        f"f1.commit={f1.commit_index} f1.visible={f1.visible_to_clients()}  "
        f"f2.commit={f2.commit_index} f2.visible={f2.visible_to_clients()}"
    )

step("append A",        lambda: leader.append("A"))
step("ship A -> f1",    lambda: leader.replicate_to(f1, 0))
step("ship A -> f2",    lambda: leader.replicate_to(f2, 0))  # quorum reached, hwm=0
step("heartbeat -> f1", lambda: leader.heartbeat_to(f1))     # f1 learns hwm=0
step("heartbeat -> f2", lambda: leader.heartbeat_to(f2))     # f2 learns hwm=0

step("append B",        lambda: leader.append("B"))
step("ship B -> f1",    lambda: leader.replicate_to(f1, 1))  # f1's ack makes hwm=1
# Careful: f1 does NOT learn hwm=1 from that message. The leader stamped the
# AppendEntries with the HWM it had *before* f1's ack arrived (still 0). f1 only
# finds out on the NEXT message — that one-round-trip lag is inherent, not a bug.
step("heartbeat -> f1", lambda: leader.heartbeat_to(f1))     # now f1 learns hwm=1
step("heartbeat -> f2", lambda: leader.heartbeat_to(f2))     # f2 still missing B, cannot expose it

# The two invariants that make follower reads safe:
# 1. A follower never exposes more than the leader has committed.
assert f1.commit_index <= leader.high_water_mark
assert f2.commit_index <= leader.high_water_mark
# 2. A follower never exposes an entry it does not physically hold — f2 is missing B,
#    so even though the cluster committed B, f2 correctly refuses to serve it.
assert "B" in leader.log and leader.high_water_mark == 1, "B should be committed"
assert f2.visible_to_clients() == ["A"], f2.visible_to_clients()
assert f1.visible_to_clients() == ["A", "B"], f1.visible_to_clients()
print("\n✔ f2 lags but never lies: it serves only what it actually has, and only up to the HWM")


Two important things to notice:

1. **`f2.visible` never shows `B`** even though the cluster has committed it - `f2` can only expose what it physically has. That is correct.
2. **The HWM always lags by one round trip.** When the leader ships entry `B` to `f1`, it
   stamps the message with the HWM it holds *at that moment* — which is still `0`, because
   `f1`'s ack has not come back yet. `f1` only learns that `B` is committed on the *next*
   message it receives. In practice that next message is either the following `AppendEntries`
   (piggybacked, free) or a heartbeat when there is no new data flowing. This is why an idle
   Raft/Kafka cluster still exchanges heartbeats: without them, the last committed entry would
   stay invisible to followers indefinitely.

## Truncation after a leadership change

Suppose the leader appended an entry `X` that no follower received, then crashed. `f1` becomes the new leader. When the old leader recovers and rejoins as a follower, it has an extra entry beyond the cluster's HWM. The new leader will tell it to **truncate** that suffix.


In [ ]:
# Leader appended X but never replicated it.
leader.append("X")
print("old leader log:", leader.log,
      "committed:", leader.log[: leader.high_water_mark + 1])

# Crash leader. f1 (which has [A, B]) becomes the new leader.
new_leader = Leader(name="L2", log=list(f1.log))
# Old leader rejoins as a follower; its log has the bogus 'X'.
old_leader_as_follower = Follower("old", log=list(leader.log))
print("rejoining follower log (before truncate):", old_leader_as_follower.log)

# The new leader has 2 entries; the rejoiner has 3. The extra one was never
# committed, so the rejoiner is told to cut its log back to the leader's length
# BEFORE it resumes fetching. (Kafka does exactly this: a replica truncates to the
# new leader's high watermark on becoming a follower. Raft achieves the same thing
# via the prevLogIndex/prevLogTerm consistency check.)
dropped = old_leader_as_follower.truncate_to(len(new_leader.log))
print(f"told to truncate to offset {len(new_leader.log)} -> dropped {dropped} entry")

# Now normal replication resumes. Re-sending A and B is a no-op, as it should be.
for off in range(len(new_leader.log)):
    new_leader.replicate_to(old_leader_as_follower, off)

print("rejoining follower log (after  truncate):", old_leader_as_follower.log)

assert old_leader_as_follower.log == ["A", "B"], old_leader_as_follower.log
# Only the uncommitted tail was removed. The committed prefix was left alone —
# re-sending A and B must be a no-op, not a rewrite.
assert old_leader_as_follower.log == new_leader.log

# Regression guard for the truncation rule itself: re-sending an entry the follower
# already holds must NOT wipe out the entries after it.
probe = Follower("probe", log=["A", "B", "C"])
probe.receive_append(0, "A", 2)
assert probe.log == ["A", "B", "C"], f"matching entry wrongly truncated the suffix: {probe.log}"
probe.receive_append(1, "B-different", 2)
assert probe.log == ["A", "B-different"], f"conflicting entry did not truncate: {probe.log}"
print("✔ truncation happens on conflict only, never on a matching re-send")


## What a leader change can and cannot do to the HWM

The truncation above is safe because `X` was never committed. The natural follow-up question —
*can a leader change ever roll the high-water mark backwards?* — is the one that decides whether
"committed" means anything at all.

It cannot, and the reason is the quorum overlap: a committed entry sits on a majority, and any
new leader needs a majority to be elected, so the two sets share at least one node. The new
leader is therefore guaranteed to already hold every committed entry.

Let's check that on every possible failover from this cluster state rather than take it on
faith: commit some entries, leave an uncommitted tail on one node, then try *every* majority
that could elect a leader and confirm none of them can lose a committed entry.

In [ ]:
from itertools import combinations

def failover_survives_commits():
    L = Leader()
    a, b, c = Follower("a"), Follower("b"), Follower("c"), 

    # Commit two entries on a majority (leader + a + b of 4 nodes -> quorum 3).
    L.quorum = 3
    for entry in ("e0", "e1"):
        off = L.append(entry)
        for f in (a, b):
            L.replicate_to(f, off)
    committed = L.log[: L.high_water_mark + 1]

    # An uncommitted tail exists only on the leader (and, say, on c which lags behind).
    L.append("e2-uncommitted")

    results = {}
    # Every possible election among the surviving nodes needs a majority (3 of 4).
    survivors = {"a": a, "b": b, "c": c}
    for group in combinations(survivors, 2):          # leader died; 2 followers + the vote
        # The elected leader is the most up-to-date node in the voting group.
        elected = max((survivors[n] for n in group), key=lambda f: len(f.log))
        results[group] = elected.log[: len(committed)] == committed
    return committed, results

committed, results = failover_survives_commits()
print("committed entries:", committed)
for group, ok in results.items():
    print(f"  electing from {group}: committed prefix intact? {ok}")

# c never received anything, so any quorum must include a or b — and both hold the
# committed prefix. There is no majority that can lose a committed entry.
assert all(results.values()), results
print("\n✔ no reachable failover rolls the high-water mark backwards")
print("  (that is the quorum-overlap argument, checked instead of asserted in prose)")

`X` is gone - and that is the **right** outcome, because it was never committed and no client was ever told it succeeded.

## Where this shows up in real systems

| System | Name they use | Notes |
|---|---|---|
| **Apache Kafka** | *High Watermark* | Per-partition; equals the smallest log-end-offset of all in-sync replicas (ISR). Consumers can only read up to it. |
| **Raft** (etcd, Consul, CockroachDB, TiKV) | `commitIndex` | Leader advances `commitIndex` once a log entry is on a majority; followers learn the new value via `AppendEntries` (which doubles as the heartbeat). |
| **MongoDB** | *majority commit point* | Tracked across replica set members; `readConcern: "majority"` exposes only data at-or-below it. |
| **PostgreSQL streaming replication** | `flush_lsn` / `apply_lsn` | Synchronous replicas only ack a transaction once it is flushed; clients with `synchronous_commit=on` wait for that ack - the same idea, expressed in WAL byte offsets. |
| **ZooKeeper / ZAB** | *committed zxid* | Leader proposes, a quorum of followers ack, then leader broadcasts COMMIT. |

The vocabulary differs but the pattern is the same: **do not expose data until a quorum of nodes durably has it**.

## Final takeaways

- The HWM is the boundary between *durable & visible* and *tentative & hidden*.
- It moves forward only as quorum acks arrive.
- Followers learn it from the leader (via heartbeats / piggybacked metadata) and may need to **truncate** when they have entries past the HWM after a leader change.
- Whenever you see "majority writes", "ISR", `commitIndex`, or "majority read concern", a high-water mark is doing the work behind the scenes.

## Try it yourself

- Change `quorum` to `3` in `Leader` and add a third follower. How does the HWM change?
- Make `replicate_to` randomly drop messages and add a retry loop. The HWM should still be safe - it just advances slower.
- Add a `read(offset)` method to `Follower` that **refuses** reads above `commit_index`. That is the consumer-side guardrail that makes follower reads safe.
